# Movies Pipeline — TMDB API


**You need:** A free TMDB API key from https://www.themoviedb.org/settings/api


In [3]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [4]:
import requests
import pandas as pd
from datetime import datetime

TMDB_API_KEY = "30df9e6cfb29fb0b594f7193054b1c1e"  


---
## Step 1: Extract movies from TMDB API


In [5]:
all_movies = []

for page in range(1, 51):                          # 50 pages = ~1000 movies
    r = requests.get(
        "https://api.themoviedb.org/3/movie/popular",
        params={"api_key": TMDB_API_KEY, "page": page},
        verify=False
    )
    r.raise_for_status()
    all_movies.extend(r.json()["results"])
    if page % 10 == 0:
        print(f"  Page {page} done...")

print(f"\nExtracted {len(all_movies)} movies")


  Page 10 done...
  Page 20 done...
  Page 30 done...
  Page 40 done...
  Page 50 done...

Extracted 1000 movies


---
## Step 2: Clean and save to CSV (Bronze)


In [6]:
rows = []
for m in all_movies:
    rows.append({
        "movie_id":     int(m["id"]),
        "title":        m["title"],
        "release_date": m.get("release_date", ""),
        "popularity":   round(float(m["popularity"]), 2),
        "vote_average": round(float(m["vote_average"]), 2),
        "vote_count":   int(m["vote_count"]),
        "language":     m["original_language"],
        "overview":     m.get("overview", "")[:200],
        "extracted_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    })

df = pd.DataFrame(rows)
df.to_csv("movies_raw.csv", index=False)
print(f"Saved {len(df)} movies to movies_raw.csv")
df.head()


Saved 1000 movies to movies_raw.csv


,movie_id,title,release_date,popularity,vote_average,vote_count,language,overview,extracted_at
0,969681,Spider-Man: Brand New Day,2026-07-29,655.44,8.47,5734,en,Fighting crime full-time as Spider-Man in a wo...,2026-10-19 09:07:31
1,1148365,Matchbox the Movie,2026-10-05,363.66,7.81,415,en,When a group of former childhood friends recon...,2026-10-19 09:07:31
2,1291595,Insidious: Out of the Further,2026-08-19,358.60,7.71,820,en,"Gemma, a young mother raising her daughter in ...",2026-10-19 09:07:31
3,1368337,The Odyssey,2026-07-15,342.35,8.00,4167,en,"Odysseus, the legendary King of Ithaca, embark...",2026-10-19 09:07:31
4,1058424,Hope,2026-07-15,368.50,6.91,257,ko,In the remote South Korean village of Hope Har...,2026-10-19 09:07:31


---
## Step 3: Load raw CSV


In [7]:
df = pd.read_csv("movies_raw.csv")
print(f"Loaded {len(df)} rows from movies_raw.csv")
df.head()


Loaded 1000 rows from movies_raw.csv


,movie_id,title,release_date,popularity,vote_average,vote_count,language,overview,extracted_at
0,969681,Spider-Man: Brand New Day,2026-07-29,655.44,8.47,5734,en,Fighting crime full-time as Spider-Man in a wo...,2026-10-19 09:07:31
1,1148365,Matchbox the Movie,2026-10-05,363.66,7.81,415,en,When a group of former childhood friends recon...,2026-10-19 09:07:31
2,1291595,Insidious: Out of the Further,2026-08-19,358.60,7.71,820,en,"Gemma, a young mother raising her daughter in ...",2026-10-19 09:07:31
3,1368337,The Odyssey,2026-07-15,342.35,8.00,4167,en,"Odysseus, the legendary King of Ithaca, embark...",2026-10-19 09:07:31
4,1058424,Hope,2026-07-15,368.50,6.91,257,ko,In the remote South Korean village of Hope Har...,2026-10-19 09:07:31


---
## Step 4: SCD2 — Track changes over time

**How it works:**
- First run: every movie is NEW → insert with `is_current = True`
- Later runs: compare incoming data vs current SCD2 rows
  - If `popularity`, `vote_average` or `vote_count` changed → expire old row, insert new one
  - If nothing changed → skip
  - If movie is brand new → insert


In [8]:
import os

KEY_COL    = "movie_id"
TRACK_COLS = ["popularity", "vote_average", "vote_count"]      # columns we watch for changes
SCD2_FILE  = "movies_scd2.csv"
now        = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Load existing SCD2 file if it exists, otherwise start empty
if os.path.exists(SCD2_FILE):
    scd2 = pd.read_csv(SCD2_FILE)
    print(f"Existing SCD2: {len(scd2)} rows")
else:
    scd2 = pd.DataFrame()
    print("No existing SCD2 file — starting fresh")


No existing SCD2 file — starting fresh


In [9]:
# Get current (active) rows from SCD2
if len(scd2) > 0:
    current = scd2[scd2["is_current"] == True].copy()
else:
    current = pd.DataFrame()

new_rows    = []
changed_ids = []

for _, row in df.iterrows():
    mid = row[KEY_COL]

    # Check if this movie already exists in SCD2
    match = current[current[KEY_COL] == mid] if len(current) > 0 else pd.DataFrame()

    if len(match) == 0:
        # NEW movie — insert
        new_rows.append(row)

    else:
        # EXISTING — check if tracked columns changed
        old = match.iloc[0]
        changed = False
        for col in TRACK_COLS:
            if str(row[col]) != str(old[col]):
                changed = True
                break

        if changed:
            changed_ids.append(mid)
            new_rows.append(row)

print(f"New:       {len(new_rows) - len(changed_ids)}")
print(f"Changed:   {len(changed_ids)}")
print(f"Unchanged: {len(df) - len(new_rows)}")


New:       1000
Changed:   0
Unchanged: 0


In [10]:
# Expire old rows for changed movies
if len(changed_ids) > 0 and len(scd2) > 0:
    mask = (scd2[KEY_COL].isin(changed_ids)) & (scd2["is_current"] == True)
    scd2.loc[mask, "is_current"] = False
    scd2.loc[mask, "valid_to"]   = now

# Build new SCD2 rows
inserts = pd.DataFrame(new_rows)
inserts["valid_from"]  = now
inserts["valid_to"]    = "9999-12-31 00:00:00"
inserts["is_current"]  = True

# Combine and save
scd2 = pd.concat([scd2, inserts], ignore_index=True)
scd2.to_csv(SCD2_FILE, index=False)

total   = len(scd2)
active  = len(scd2[scd2["is_current"] == True])
history = total - active
print(f"\nSCD2 saved: {total} total rows ({active} current, {history} historical)")



SCD2 saved: 1000 total rows (1000 current, 0 historical)


---
## Verify


In [11]:
scd2 = pd.read_csv(SCD2_FILE)

print(f"Total rows:      {len(scd2)}")
print(f"Current rows:    {len(scd2[scd2['is_current'] == True])}")
print(f"Historical rows: {len(scd2[scd2['is_current'] == False])}")
print(f"Unique movies:   {scd2['movie_id'].nunique()}")

# Show a sample
scd2.head(10)


Total rows:      1000
Current rows:    1000
Historical rows: 0
Unique movies:   988


,movie_id,title,release_date,popularity,vote_average,vote_count,language,overview,extracted_at,valid_from,valid_to,is_current
0,969681,Spider-Man: Brand New Day,2026-07-29,655.44,8.47,5734,en,Fighting crime full-time as Spider-Man in a wo...,2026-10-19 09:07:31,2026-10-19 09:07:46,9999-12-31 00:00:00,True
1,1148365,Matchbox the Movie,2026-10-05,363.66,7.81,415,en,When a group of former childhood friends recon...,2026-10-19 09:07:31,2026-10-19 09:07:46,9999-12-31 00:00:00,True
2,1291595,Insidious: Out of the Further,2026-08-19,358.60,7.71,820,en,"Gemma, a young mother raising her daughter in ...",2026-10-19 09:07:31,2026-10-19 09:07:46,9999-12-31 00:00:00,True
3,1368337,The Odyssey,2026-07-15,342.35,8.00,4167,en,"Odysseus, the legendary King of Ithaca, embark...",2026-10-19 09:07:31,2026-10-19 09:07:46,9999-12-31 00:00:00,True
4,1058424,Hope,2026-07-15,368.50,6.91,257,ko,In the remote South Korean village of Hope Har...,2026-10-19 09:07:31,2026-10-19 09:07:46,9999-12-31 00:00:00,True
5,1423191,Resident Evil,2026-09-16,230.13,7.30,984,en,Medical courier Bryan unwittingly finds himsel...,2026-10-19 09:07:31,2026-10-19 09:07:46,9999-12-31 00:00:00,True
6,1236045,Animals,2026-10-09,236.84,7.00,482,en,When the son of an L.A. mayoral candidate is k...,2026-10-19 09:07:31,2026-10-19 09:07:46,9999-12-31 00:00:00,True
7,1228834,The Fix,2026-09-10,237.13,6.71,65,en,Disillusioned by the end of the war in Afghani...,2026-10-19 09:07:31,2026-10-19 09:07:46,9999-12-31 00:00:00,True
8,1377237,Runner,2026-09-07,233.44,8.34,651,en,A former soldier is thrown into a brutal race ...,2026-10-19 09:07:31,2026-10-19 09:07:46,9999-12-31 00:00:00,True
9,1101412,Fall 2: Deadpoint,2026-09-01,281.62,7.26,596,en,On a perilous climb across Thailand's Mount Kw...,2026-10-19 09:07:31,2026-10-19 09:07:46,9999-12-31 00:00:00,True


---
## How to see SCD2 in action

Run this notebook again tomorrow. The popularity and vote counts will have changed
for some movies. You'll see:
- Old rows with `is_current = False` and a `valid_to` timestamp
- New rows with `is_current = True`

That's SCD2 — full history of every change.
